# F1 · Gini or entropy? Best split or random split?

**Type**: Compare · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Systematically compare the impurity criterion (`gini`, `entropy`) and the `splitter` (`best`, `random`) of a tree on three datasets (wines, tumors, digits) with a repeated cross-validation, measuring accuracy, number of leaves and training time, and establish how much these choices matter compared with the measurement noise
of a single tree.

## Dataset

`load_wine`, `load_breast_cancer`, `load_digits`. Model: `DecisionTreeClassifier(min_samples_leaf=3, random_state=0)` with the four combinations; validation `RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)`, accuracy.

## Educational purpose

Discover that the criterion (Gini or entropy) is a minor detail compared with regularization, and that the uncertainty of a single tree (the variability between folds) is of the same size as the differences between the options: do not draw conclusions from a single score.

## Task

1. For each dataset and each combination (`gini/best`, `entropy/best`, `gini/random`, `entropy/random`) compute with `cross_validate` (with `return_train_score=False`) `acc` (mean), `sd` (standard deviation across the 20 folds) and `t_fit` (mean fit time in seconds). Save `table` with index `(dataset, combination)` and the three columns.
2. For each combination train a tree on the whole dataset and save `leaves` (`pd.Series` with the same index).
3. Save `criterion_diff` (dictionary dataset → `acc(entropy/best) − acc(gini/best)`) and `max_criterion_diff` (maximum absolute value).
4. Save `random_has_more_leaves` (True if for all the datasets `gini/random` has more leaves than `gini/best`) and `typical_sd` (median of the `sd` of `table`). Answer with `criterion_matters = False` if `max_criterion_diff` is smaller than `2 * typical_sd`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

datasets = {"wine": load_wine(return_X_y=True), "tumors": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
combinations = {"gini/best": ("gini", "best"), "entropy/best": ("entropy", "best"), "gini/random": ("gini", "random"), "entropy/random": ("entropy", "random")}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)

## Your work

In [ ]:
rows = {}
leaves = {}
for name, (X, y) in datasets.items():
    for comb, (criterion, splitter) in combinations.items():
        model = DecisionTreeClassifier(criterion=criterion, splitter=splitter, min_samples_leaf=3, random_state=0)
        ...
table = pd.DataFrame(rows).T
leaves = pd.Series(leaves)
table.round(3)

In [ ]:
criterion_diff = {}
max_criterion_diff = None
random_has_more_leaves = None
typical_sd = None
criterion_matters = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`r = cross_validate(model, X, y, cv=cv)`; `acc = r["test_score"].mean()`, `sd = r["test_score"].std()`, `t_fit = r["fit_time"].mean()`. Leaves: `model.fit(X, y).get_n_leaves()`. Key: `leaves[(name, comb)] = ...` and `rows[(name, comb)] = {...}`.'):
    check(table.shape == (12, 3) and list(table.columns) == ["acc", "sd", "t_fit"] and len(leaves) == 12, "12 rows (3 datasets x 4 combinations) and three columns; 12 leaf counts", "acc, sd, t_fit")
    check(set(criterion_diff) == {"wine", "tumors", "digits"} and close(max_criterion_diff, max(abs(v) for v in criterion_diff.values())) and close(criterion_diff["wine"], table.loc[("wine", "entropy/best"), "acc"] - table.loc[("wine", "gini/best"), "acc"]), "differences between criteria computed", "acc(entropy/best) - acc(gini/best)")
    check(max_criterion_diff < 0.04, "the difference between the two criteria is always below 4 points", "compare the acc values")
    check(random_has_more_leaves is True and (leaves[[("wine", "gini/random"), ("tumors", "gini/random"), ("digits", "gini/random")]].to_numpy() > leaves[[("wine", "gini/best"), ("tumors", "gini/best"), ("digits", "gini/best")]].to_numpy()).all(), "the random splitter gives more leaves, on all the datasets", "leaves of gini/random against gini/best")
    check(0.015 < typical_sd < 0.06 and criterion_matters is False, "the standard deviation across folds (2-6 points) is of the same order as the differences: the criterion does not matter in a distinguishable way", "median of the sd")
done()

---
The commented solution is in `solutions/F_compare/F1_gini_entropy_and_random_splitter.ipynb`. Open it after you pass the check, or after you have really tried.